In [ ]:
# 🧠 Wiz Sports DFS Optimizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DJWiz69/wiz-sports-pro-optimizer/blob/main/wiz-sports-pro-optimizer.ipynb)

# Wiz Sports Pro Optimizer — Jupyter Notebook Edition (Full Power Mode)

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from itertools import combinations
import base64
from io import StringIO
import random
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# ----------------------------
# STEP 2: Optimizer Controls
# ----------------------------

# Sliders and input fields for filters and projections
salary_slider = widgets.IntRangeSlider(value=[5000, 9000], min=3000, max=12000, step=500,
                                       description='Salary Range:', continuous_update=False)
value_threshold = widgets.FloatSlider(value=4.0, min=1.0, max=10.0, step=0.1,
                                      description='Min Value:', continuous_update=False)
num_lineups = widgets.IntSlider(value=5, min=1, max=20, step=1, description='Lineups:')

# Display controls
display(salary_slider, value_threshold, num_lineups)

# ----------------------------
# STEP 3: Generate Lineups
# ----------------------------

def run_optimizer(*args):
    try:
        filtered = df.copy()
        min_salary, max_salary = salary_slider.value
        filtered = filtered[(filtered['Salary'] >= min_salary) & (filtered['Salary'] <= max_salary)]
        filtered['Value'] = filtered['Projection'] / filtered['Salary'] * 1000
        filtered = filtered[filtered['Value'] >= value_threshold.value]
        top_players = filtered.sort_values(by='Projection', ascending=False).head(20)

        lineups = []
        for _ in range(num_lineups.value):
            lineup = top_players.sample(n=5, replace=False).sort_values(by='Salary')
            total_proj = lineup['Projection'].sum()
            total_salary = lineup['Salary'].sum()
            lineups.append((lineup, total_proj, total_salary))

        for i, (lineup, proj, salary) in enumerate(lineups):
            print(f"\nLineup {i+1} — Total Projection: {proj:.2f}, Total Salary: ${salary}")
            display(lineup[['Player', 'Position', 'Salary', 'Projection']])

    except Exception as e:
        print("⚠️ Error generating lineups:", e)

run_button = widgets.Button(description='Run Wiz Full Optimizer', button_style='success')
run_button.on_click(run_optimizer)
display(run_button)

# ----------------------------
# STEP 1: Upload Projections
# ----------------------------

upload_widget = widgets.FileUpload(accept='.csv,.xlsx', multiple=False)
display(upload_widget)

def handle_upload(change):
    global df  # Make accessible to future steps
    uploaded_file = next(iter(upload_widget.value.values()))
    content = uploaded_file['content']
    if uploaded_file['metadata']['name'].endswith('.csv'):
        df = pd.read_csv(StringIO(content.decode('utf-8')))
    elif uploaded_file['metadata']['name'].endswith('.xlsx'):
        from io import BytesIO
        df = pd.read_excel(BytesIO(content))
    else:
        print("Unsupported file type.")
        return
    display(df)

upload_widget.observe(handle_upload, names='value')
